In [1]:
%pip install rank-bm25
from rank_bm25 import BM25Okapi 
from chroma import ma_collection
import string
import re
import numpy as np 


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Chemin utilisé : C:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\chroma_3gpp_rel18
Existe : True

Contenu du dossier :
- ._chroma.sqlite3
- ._eae17378-2ffa-4d50-9800-01eaa05f2742
- chroma.sqlite3
- eae17378-2ffa-4d50-9800-01eaa05f2742

Nombre de collections : 1
[Collection(name=3gpp_rel18_collection)]
Collection(name=3gpp_rel18_collection)

Nombre de chunk : 562865
{'ids': ['2865084', '2865085', '2865086', '2865087', '2865088', '2865089', '2865090', '2865091', '2865092', '2865093'], 'embeddings': array([[-0.0092853 ,  0.01435731, -0.00187875, ...,  0.07671405,
         0.05635262, -0.00273231],
       [ 0.05003922,  0.0073945 , -0.01688451, ...,  0.00345017,
         0.04526629,  0.02224259],
       [-0.00206988,  0.02758421,  0.03543698, ..., -0.04465876,
         0.00797065,  0.01978247],
       ...,
       [-0.07843515,  0.01852478,  0.04259703, ...,  0.04828352,
         0.0727049 ,  0.02867837],
       [-0.09269179,  0.03357768,  0.0112438 , ...,  0.0197269

In [2]:
from rank_bm25 import BM25Okapi
import re

# récupérer les documents
data = ma_collection.get(
    limit=5000,
    include=["documents"]
)

corpus = data["documents"]

# fonction pour découper le texte en mots
def tokenize(text):
    return re.findall(r"[A-Za-z0-9]+(?:[._-][A-Za-z0-9]+)*", text.lower())

# tokeniser tous les documents
tokenized_corpus = [tokenize(doc) for doc in corpus]

# créer BM25
bm25 = BM25Okapi(tokenized_corpus)

Test

In [3]:
question = "What is a Change Request?"

question_tokenized = tokenize(question)

scores = bm25.get_scores(question_tokenized)

Top K

In [4]:
top_indices_bm25 = np.argsort(scores)[::-1][:5]

for i in top_indices_bm25:
    print("Score :", scores[i])
    print(corpus[i])
    print()

Score : 13.048107483030694
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - + | pcr | pseudo - change - request : similar to a change | | | request but has no cr number and is intended | | | to propose new or revised text for inclusion | | | in 3gpp tss or trs not yet under change |

Score : 12.696033919347556
Section: 4.6	Change Request regime > 4.6.2	Change Request forms
Table Analysis:
Caption: Table 4A: Categories of Change Requests
Data:
| Category | Meaning                    | Remarks                    |
+----------+----------------------------+----------------------------+
|          |                            | > leads to incorrect       |
|          |                            | > operation of the         |
|          |                            | > system); or              |
|          |                            | >                          |
|          |                            | > 2 to correct an          |
|          |                            | > a

In [5]:
def rrf_fusion(bm25_indices, bge_indices, k=60, top_k=5):

    scores = {}

    # BM25
    for rank, idx in enumerate(bm25_indices, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    # BGE
    for rank, idx in enumerate(bge_indices, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    # Trier du meilleur score au plus faible
    ranking_final = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranking_final[:top_k]